# Build 7 -- Ensembling / Blending

Kaggle Playground Series S6E8 -- Predicting Smartphone Addiction

**Objective:** can combining genuinely different strong models beat E010
(the frozen Build 6 control, CV mean 0.96499, public LB 0.96653), via a
disciplined diversity-check -> simple-blend -> weighted-blend ->
(rank averaging / stacking only if justified) pipeline? Ensembling is not
pursued for its own sake -- every step requires evidence before
proceeding to the next.

**Frozen for this build:** every base model's feature set and
hyperparameters (E010's XGBoost config, E008's CatBoost config, E006's
pre-tuned XGBoost config, E003's LightGBM config all unchanged from
Builds 3/4/6). Only combination of already-frozen models is in scope.

**Explicitly out of scope:** new features, retuning any base model,
final submission selection, project consolidation -- all deferred to
Build 8/9 or later.

In [1]:
import sys
from itertools import combinations
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from src.config import EXPERIMENTS_DIR, ID_COLUMN, TARGET_COLUMN, TEST_PATH, TRAIN_PATH
from src.ensembling import (
    fold_auc_breakdown,
    load_fold_assignments,
    load_oof,
    load_test_pred,
    pairwise_diversity,
    rank_blend,
    weighted_blend,
)

pd.set_option("display.max_columns", 50)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
ids = train[ID_COLUMN]
test_ids = test[ID_COLUMN]
y = train[TARGET_COLUMN]

print("train shape:", train.shape, "test shape:", test.shape)

train shape: (691369, 14) test shape: (296302, 13)


## 2. Candidate model inventory

Pulled directly from `experiments/experiments.csv` -- no hardcoded
numbers. See `docs/BUILD_HISTORY.md`'s Build 7 entry for the full
dominated/redundant/diversity reasoning behind which experiments made
the active pool.

In [2]:
experiments = pd.read_csv(EXPERIMENTS_DIR / "experiments.csv")

CANDIDATE_IDS = ["E010", "E008", "E006", "E003", "E002", "E001"]
cols = ["experiment_id", "model", "feature_set", "cv_mean", "cv_std", "public_lb"]
inventory = (
    experiments.loc[experiments["experiment_id"].isin(CANDIDATE_IDS), cols]
    .set_index("experiment_id")
    .loc[CANDIDATE_IDS]
)
inventory

,model,feature_set,cv_mean,cv_std,public_lb
experiment_id,,,,,
E010,XGBClassifier,raw + screen_residual,0.96499,0.00051,0.96653
E008,CatBoostClassifier,raw + screen_residual,0.96104,0.00055,NaN
E006,XGBClassifier,raw + screen_residual,0.96445,0.00056,0.96608
E003,LGBMClassifier,raw_predictors,0.96106,0.00113,NaN
E002,CatBoostClassifier,raw_predictors,0.96040,0.00051,0.96151
E001,LogisticRegression,raw_predictors,0.91149,0.00081,0.91358


**Active pool: E010 (primary), E008 (CatBoost, diversity), E006
(pre-tuned XGBoost, suspected redundant -- tested anyway), E003
(LightGBM, diversity).** E001 and E002 are shown for context but
excluded: E002 is dominated by E008 (same model, worse feature set);
E001 is ~5.3pt CV AUC below E010, too weak to plausibly help. E005,
E007, E009 (not shown) are excluded as rejected-feature XGBoost variants
already redundant with E006/E010.

## 3. OOF/test artifact validation (Phase 0)

No experiment in this repository had ever persisted a raw per-row OOF or
test prediction array before this build -- only scalar correlation
values survived past their originating notebook run. All four active
candidates' predictions were regenerated by reconstructing each
experiment's *exact* frozen configuration
(`src/regenerate_ensemble_predictions.py`, run once in the background)
and are loaded here from `outputs/oof_predictions/` and
`outputs/test_predictions/`. `src.ensembling.load_oof`/`load_test_pred`
validate id alignment, row count, and probability bounds on every load
-- never blended by positional assumption alone.

In [3]:
ACTIVE_CANDIDATES = ["E010", "E008", "E006", "E003"]

folds = load_fold_assignments(ids)

oof = {exp: load_oof(exp, ids) for exp in ACTIVE_CANDIDATES}
test_pred = {exp: load_test_pred(exp, test_ids) for exp in ACTIVE_CANDIDATES}

recorded_cv_mean = experiments.set_index("experiment_id").loc[ACTIVE_CANDIDATES, "cv_mean"]
for exp in ACTIVE_CANDIDATES:
    reconstructed = roc_auc_score(y, oof[exp])
    recorded = float(recorded_cv_mean[exp])
    assert abs(reconstructed - recorded) < 2e-4, f"{exp}: {reconstructed:.5f} vs {recorded:.5f}"
    print(f"{exp}: OOF-recomputed AUC {reconstructed:.5f} matches recorded cv_mean {recorded:.5f}")

print(f"\nAll {len(ACTIVE_CANDIDATES)} candidates' OOF ({len(ids)} rows) and test "
      f"({len(test_ids)} rows) predictions loaded, id-validated, bounds-checked.")

E010: OOF-recomputed AUC 0.96499 matches recorded cv_mean 0.96499


E008: OOF-recomputed AUC 0.96104 matches recorded cv_mean 0.96104


E006: OOF-recomputed AUC 0.96444 matches recorded cv_mean 0.96445


E003: OOF-recomputed AUC 0.96106 matches recorded cv_mean 0.96106

All 4 candidates' OOF (691369 rows) and test (296302 rows) predictions loaded, id-validated, bounds-checked.


## 4. Prediction correlation / diversity analysis (Phase 1)

Pearson and Spearman alone are not sufficient evidence of redundancy --
two models can be highly correlated overall while still disagreeing near
the ranking boundaries that matter most for ROC AUC. `mean_abs_diff` and
top/bottom-decile disagreement are computed alongside.

In [4]:
diversity_rows = [
    pairwise_diversity(a, oof[a].to_numpy(), b, oof[b].to_numpy())
    for a, b in combinations(ACTIVE_CANDIDATES, 2)
]
diversity_df = pd.DataFrame(diversity_rows).round(4)
diversity_df

,model_a,model_b,pearson,spearman,mean_abs_diff,top_decile_disagreement,bottom_decile_disagreement
0,E010,E008,0.9849,0.9847,0.0357,0.2805,0.1160
1,E010,E006,0.9967,0.9974,0.0151,0.1036,0.0563
2,E010,E003,0.9849,0.9833,0.0319,0.3277,0.1086
3,E008,E006,0.9877,0.9865,0.0315,0.2657,0.1070
4,E008,E003,0.9898,0.9783,0.0254,0.4246,0.1004
5,E006,E003,0.9872,0.9842,0.0282,0.3263,0.1007


**Findings:** E006 is by far the most redundant with E010 (0.9967
Pearson, lowest disagreement of every metric, every pair). E008 and E003
are each genuinely diverse from E010 (~0.985 Pearson but 28-33%
top-decile disagreement) and from each other (42.5% top-decile
disagreement -- the highest of any pair measured, confirming LightGBM
adds real diversity beyond CatBoost). Both are individually strong
enough to matter (CV ~0.961, ~0.004 below E010) -- this is not the "weak
model with low correlation" pattern the Build 7 brief warns against.

## 5. Equal-weight probability blends (Phase 2)

50/50 OOF blends for the four pairs the diversity analysis justifies
testing. Classification: Clear >= +0.0005 (Build 6's own established
meaningful-gain bar), Marginal > +0.0001, Flat within +/-0.0001, Worse
< -0.0001.

In [5]:
e010_solo_auc = roc_auc_score(y, oof["E010"])

def classify(delta: float) -> str:
    if delta >= 0.0005:
        return "Clear"
    if delta > 0.0001:
        return "Marginal"
    if delta >= -0.0001:
        return "Flat"
    return "Worse"

PAIRS = [("E010", "E008"), ("E010", "E003"), ("E010", "E006"), ("E008", "E003")]

equal_weight_rows = []
for a, b in PAIRS:
    blend = weighted_blend({a: oof[a].to_numpy(), b: oof[b].to_numpy()}, {a: 0.5, b: 0.5})
    auc = roc_auc_score(y, blend)
    delta = auc - e010_solo_auc
    equal_weight_rows.append(
        {"pair": f"{a}+{b}", "oof_auc": round(auc, 5), "delta_vs_e010": round(delta, 5), "decision": classify(delta)}
    )

print(f"E010 solo OOF AUC: {e010_solo_auc:.5f}\n")
pd.DataFrame(equal_weight_rows)

E010 solo OOF AUC: 0.96499



,pair,oof_auc,delta_vs_e010,decision
0,E010+E008,0.96414,-0.00085,Worse
1,E010+E003,0.96417,-0.00082,Worse
2,E010+E006,0.96495,-0.00004,Flat
3,E008+E003,0.96187,-0.00312,Worse


Every 50/50 blend is Worse or Flat vs E010 alone -- expected and
mechanical: averaging a 0.96499 model equally with a ~0.961 model
necessarily pulls the mean down, regardless of how genuinely diverse the
weaker component's errors are. This does not contradict Section 4's
diversity findings; it means 50/50 is the wrong test of whether that
diversity is exploitable. The weighted grid below is the real test.

## 6. Weighted probability blends (Phase 3)

Coarse grid only (no continuous optimizer): E010 weight in
{0.5, 0.6, 0.7, 0.8, 0.9, 0.95} for the three E010 pairs; E008 weight in
{0.3, 0.4, 0.5, 0.6, 0.7} for E008+E003. Summarized to the best point per
pair -- see `outputs/ensemble_results.csv` for the full grid.

In [6]:
E010_WEIGHT_GRID = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
E008_WEIGHT_GRID = [0.3, 0.4, 0.5, 0.6, 0.7]

weighted_summary = []
for secondary in ["E008", "E003", "E006"]:
    best_w, best_auc = None, -1.0
    for w in E010_WEIGHT_GRID:
        blend = weighted_blend(
            {"E010": oof["E010"].to_numpy(), secondary: oof[secondary].to_numpy()},
            {"E010": w, secondary: 1 - w},
        )
        auc = roc_auc_score(y, blend)
        if auc > best_auc:
            best_w, best_auc = w, auc
    delta = best_auc - e010_solo_auc
    weighted_summary.append(
        {"pair": f"E010+{secondary}", "best_e010_weight": best_w,
         "oof_auc": round(best_auc, 5), "delta_vs_e010": round(delta, 5), "decision": classify(delta)}
    )

best_w8, best_auc8 = None, -1.0
for w in E008_WEIGHT_GRID:
    blend = weighted_blend(
        {"E008": oof["E008"].to_numpy(), "E003": oof["E003"].to_numpy()}, {"E008": w, "E003": 1 - w}
    )
    auc = roc_auc_score(y, blend)
    if auc > best_auc8:
        best_w8, best_auc8 = w, auc
delta8 = best_auc8 - e010_solo_auc
weighted_summary.append(
    {"pair": "E008+E003", "best_e010_weight": f"E008={best_w8}",
     "oof_auc": round(best_auc8, 5), "delta_vs_e010": round(delta8, 5), "decision": classify(delta8)}
)

pd.DataFrame(weighted_summary)

,pair,best_e010_weight,oof_auc,delta_vs_e010,decision
0,E010+E008,0.95,0.96500,0.00001,Flat
1,E010+E003,0.95,0.96500,0.00001,Flat
2,E010+E006,0.8,0.96503,0.00004,Flat
3,E008+E003,E008=0.5,0.96187,-0.00312,Worse


No weight, for any pair, reaches the +0.0005 Clear bar. E010+E008 and
E010+E003 both converge toward E010's own score from below as E010's
weight rises, plateauing at essentially E010 rather than exceeding it.
E010+E006's plateau (+0.00002 to +0.00004 across weights 0.6-0.9) is
well inside the ~0.0005 single-fold noise band established in Build 6.
E008+E003 never approaches E010 at any weight. This is a broad, stable
*null* result, not an isolated spike -- the search did not overfit to
noise, it consistently found nothing.

## 7. Three-model blend (Phase 4) -- explicitly skipped

The brief's own gate requires both E010+E008 and E010+E003 to show real
two-model gains before attempting a trio. Neither did (both plateau at
Flat). Adding a third component on top of two null results would not
manufacture a gain neither pairwise combination could find, so this
phase was not attempted.

## 8. Rank averaging (Phase 5)

Percentile-rank blend (`src.ensembling.rank_blend`) at representative
weights, compared directly against the probability blend at the same
weights.

In [7]:
rank_rows = []
for secondary in ["E008", "E003", "E006"]:
    for w in [0.5, 0.8, 0.9]:
        prob_blend = weighted_blend(
            {"E010": oof["E010"].to_numpy(), secondary: oof[secondary].to_numpy()},
            {"E010": w, secondary: 1 - w},
        )
        rank_blended = rank_blend(
            {"E010": oof["E010"].to_numpy(), secondary: oof[secondary].to_numpy()},
            {"E010": w, secondary: 1 - w},
        )
        prob_auc = roc_auc_score(y, prob_blend)
        rank_auc = roc_auc_score(y, rank_blended)
        rank_rows.append({
            "pair": f"E010+{secondary}", "e010_weight": w,
            "probability_auc": round(prob_auc, 5), "rank_auc": round(rank_auc, 5),
            "rank_minus_prob": round(rank_auc - prob_auc, 5),
        })

pd.DataFrame(rank_rows)

,pair,e010_weight,probability_auc,rank_auc,rank_minus_prob
0,E010+E008,0.5,0.96414,0.96408,-0.00007
1,E010+E008,0.8,0.96489,0.96488,-0.00001
2,E010+E008,0.9,0.96498,0.96498,-0.00000
3,E010+E003,0.5,0.96417,0.96418,0.00001
4,E010+E003,0.8,0.96489,0.96491,0.00002
5,E010+E003,0.9,0.96498,0.96499,0.00001
6,E010+E006,0.5,0.96495,0.96495,-0.00000
7,E010+E006,0.8,0.96503,0.96503,-0.00000
8,E010+E006,0.9,0.96502,0.96502,-0.00000


Rank averaging matches probability blending within 0.00001 at every
point tested -- no calibration difference between these four models is
large enough for rank transformation to matter. **Not adopted.**

## 9. Fold-level blend stability (Phase 6)

For the single best-surviving candidate (E010+E006 at 0.80/0.20
probability weights, the least-negative point found across every blend
tested), per-fold AUC compared directly against E010's own folds.

In [8]:
e010_fold_scores = fold_auc_breakdown(oof["E010"].to_numpy(), y, folds)

best_blend = weighted_blend(
    {"E010": oof["E010"].to_numpy(), "E006": oof["E006"].to_numpy()}, {"E010": 0.8, "E006": 0.2}
)
blend_fold_scores = fold_auc_breakdown(best_blend, y, folds)

stability = pd.DataFrame({
    "fold": range(1, 6),
    "E010": np.round(e010_fold_scores, 5),
    "E010+E006 (0.8/0.2)": np.round(blend_fold_scores, 5),
})
stability["delta"] = np.round(stability["E010+E006 (0.8/0.2)"] - stability["E010"], 5)

print(f"E010 mean/std:  {np.mean(e010_fold_scores):.6f} / {np.std(e010_fold_scores):.6f}")
print(f"Blend mean/std: {np.mean(blend_fold_scores):.6f} / {np.std(blend_fold_scores):.6f}")
print(f"Folds improved: {sum(b > a for a, b in zip(e010_fold_scores, blend_fold_scores))}/5\n")
stability

E010 mean/std:  0.964995 / 0.000508
Blend mean/std: 0.965034 / 0.000518
Folds improved: 5/5



,fold,E010,E010+E006 (0.8/0.2),delta
0,1,0.96429,0.96433,0.00004
1,2,0.96500,0.96503,0.00003
2,3,0.96509,0.96513,0.00004
3,4,0.96585,0.96591,0.00006
4,5,0.96475,0.96477,0.00002


5/5 folds improved, but every improvement is a fraction of the noise
band (+0.00002 to +0.00006) and CV std is essentially unchanged.
Consistent direction across all 5 folds rules out a single-fold fluke,
but consistency of a microscopic effect is still a microscopic effect --
the expected signature of averaging two highly correlated models from
the same architecture family (mild variance reduction, no real
information gain), not a usable ensemble.

## 10. Stacking decision gate (Phase 8) -- rejected

The weighted grid in Section 6 already performed an exhaustive linear
search over every pairwise combination of these four candidates and
found no region above E010 for any pair. A `LogisticRegression`
meta-model over the same probability inputs is itself a linear
combination of them -- it cannot discover a solution that dense grid
coverage of that exact same linear space did not already surface.
Implementing leakage-safe nested-CV stacking machinery here would add
real complexity to re-confirm a null result already established more
simply and more transparently. Per the Build 7 brief's own allowance,
"Stacking rejected as unnecessary" is a fully valid outcome.

## 11. Optional stacker

Not implemented, per the Section 10 decision.

## 12. Submission candidates (Phase 7, executed after the stacking gate)

**No Build 7 submission.** Confirmed with the user rather than assumed:
since no blend beat E010 in OOF terms, and the brief's own rule is not
to submit dominated blends, E010's standing public LB (0.96653) remains
the reference. No new row was added to `experiments/experiments.csv` --
none of the eleven blend/rank trials in `outputs/ensemble_results.csv`
cleared the bar for a "formal experiment" (an adopted configuration with
a submission), mirroring Build 6's precedent of keeping screening-only
trials out of `experiments.csv`.

## 13. Build 7 conclusions

- **No ensemble beats E010.** Across equal-weight, weighted, and
  rank-averaged blending of every candidate pair (and the
  explicitly-skipped three-model trio), the frozen E010 configuration
  (CV mean 0.96499, public LB 0.96653) remains the best available model.
- **E008 (CatBoost) and E003 (LightGBM) are both genuinely diverse from
  E010** (~0.985 Pearson, 28-33% top-decile disagreement) **and from
  each other** (42.5% top-decile disagreement, the highest of any pair
  measured) **but individually too far below E010's strength (~0.004
  AUC) for that diversity to net a gain** once weighted correctly to
  avoid dragging the blend toward the weaker component.
- **E006 (pre-tuned XGBoost) is too redundant with E010** (0.9967
  Pearson, lowest disagreement of every pair by a wide margin) to
  contribute anything beyond noise-level variance reduction (+0.00002 to
  +0.00004 per fold, 5/5 folds, CV std unchanged) -- this formally
  answers the brief's question of whether the pre-tuned XGBoost adds
  anything: no.
- **Rank averaging offers no advantage over probability averaging** for
  this candidate set -- results matched within 0.00001 at every point
  tested.
- **Three-model blending was not justified** -- neither constituent pair
  showed a real two-model gain, so the gate for attempting a trio was
  never met.
- **Stacking was rejected as unnecessary** -- the weighted grid already
  exhaustively searched the linear-combination space a logistic
  meta-model would search, and found nothing there.
- **No Build 7 Kaggle submission** -- E010's public LB (0.96653) stands
  unchanged as the best result entering Build 8.
- This is not a failed build: it is a disciplined, evidence-backed
  answer to the Build 7 question ("can we improve beyond the best
  single-model result by combining genuinely different strong models in
  a controlled, validation-backed way?") -- **no**, not with this
  candidate pool, and the evidence trail
  (`outputs/ensemble_prediction_correlations.csv`,
  `outputs/ensemble_results.csv`, persisted OOF/test predictions) means
  this does not need to be re-investigated from scratch later.